In [1]:
"""
SEED-IV -- Multi-Scale Spatial-Temporal Masked Autoencoder (Fast FP32)
===================================================================

OPTIMIZATIONS & UPDATES INCLUDED:
  1. Cleaned Output Structure: "L" rungs and 5-Fold evaluation removed.
     Strictly runs the Leave-One-Subject-Out (LOSO) evaluation protocol.
  2. Single Unified Autoencoder: STMAE handles topology and reconstruction.
  3. VRAM Pre-loading: Dataset (X_pt) stays on GPU to eliminate PCIe lag.
  4. cuDNN Benchmarking: Hardware-level convolution acceleration enabled.
"""

import os
import time
import math
import random
import copy
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.io as sio
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")

# Hardware-level optimization for static-size convolutions in MSC-TimesNet
torch.backends.cudnn.benchmark = True

# =====================================================================
# CONFIGURATION
# =====================================================================
DATA_DIR = "/kaggle/input/datasets/phhasian0710/seed-iv/eeg_feature_smooth"
OUTPUT_DIR = "/kaggle/working/seediv_fast"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SESSIONS_TO_SKIP = []           # session 3 INCLUDED
FEATURE_TYPE = "both"           # "both" | "de" | "psd"
NORMALIZATION_METHOD = "subject_session"

NUM_CHANNELS = 62
NUM_EMOTIONS = 4
TRIALS_PER_SESSION = 24
DE_KEY_PREFIX = "de_LDS"
PSD_KEY_PREFIX = "psd_LDS"

# ---- Stage A : STMAE (Single Autoencoder) ----
EMBEDDING_SIZE = 32
AUTOENCODER_HIDDEN_SIZE = 64
AUTOENCODER_HEADS = 4
AUTOENCODER_LAYERS = 3
AUTOENCODER_EPOCHS = 30
AUTOENCODER_LR = 1e-3
AUTOENCODER_BATCH_SIZE = 256
RANDOM_MASK_FRACTION = 0.40
REGION_MASK_CHANCE = 0.60

# ---- Stage B : sequences ----
WINDOW_LENGTH = 10

# ---- Stage C : MSC-TimesNet ----
CLASSIFIER_HIDDEN_SIZE = 128
CLASSIFIER_BLOCKS = 2
TOP_FREQUENCIES = 3
CLASSIFIER_FEEDFORWARD_SIZE = 256
CLASSIFIER_HEADS = 4
CLASSIFIER_DROPOUT = 0.3

# ---- classifier training ----
MAX_EPOCHS = 60
MIN_EPOCHS = 15
PATIENCE_EPOCHS = 12
LEARNING_RATE = 1e-3
ENCODER_LR_SCALE = 0.1   
BATCH_SIZE = 128
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 5
GRADIENT_CLIP = 1.0
LABEL_SMOOTHING = 0.05
VALIDATION_FRACTION = 0.2
MIXUP_STRENGTH = 0.2
CHANNEL_DROPOUT_RATE = 0.1

RECALIBRATE_BATCHNORM = True
RANDOM_SEED = 42

RUN_LOSO = True     # Runs the primary LOSO evaluation block

SEEDS_PER_PROTOCOL = {
    "loso_subject": [42, 1337, 2024],
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRIAL_LABELS_BY_SESSION = {
    1: [1, 2, 3, 0, 2, 0, 0, 1, 0, 1, 2, 1, 1, 1, 2, 3, 2, 2, 3, 3, 0, 3, 0, 3],
    2: [2, 1, 3, 0, 0, 2, 0, 2, 3, 3, 2, 3, 2, 0, 1, 1, 2, 1, 0, 3, 0, 1, 3, 1],
    3: [1, 2, 2, 1, 3, 3, 3, 1, 1, 2, 1, 0, 2, 3, 3, 0, 2, 3, 0, 0, 2, 0, 1, 0],
}

CHANNEL_NAMES = [
    "FP1", "FPZ", "FP2", "AF3", "AF4", "F7", "F5", "F3", "F1", "FZ",
    "F2", "F4", "F6", "F8", "FT7", "FC5", "FC3", "FC1", "FCZ", "FC2",
    "FC4", "FC6", "FT8", "T7", "C5", "C3", "C1", "CZ", "C2", "C4",
    "C6", "T8", "TP7", "CP5", "CP3", "CP1", "CPZ", "CP2", "CP4", "CP6",
    "TP8", "P7", "P5", "P3", "P1", "PZ", "P2", "P4", "P6", "P8",
    "PO7", "PO5", "PO3", "POZ", "PO4", "PO6", "PO8", "CB1", "O1", "OZ",
    "O2", "CB2",
]
assert len(CHANNEL_NAMES) == NUM_CHANNELS

REPORT = []


def seed_everything(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =====================================================================
# SCALP GEOMETRY (10-20 layout)
# =====================================================================
def build_scalp_coords():
    rows = [
        (["FP1", "FPZ", "FP2"], 0.95),
        (["AF3", "AF4"], 0.80),
        (["F7", "F5", "F3", "F1", "FZ", "F2", "F4", "F6", "F8"], 0.62),
        (["FT7", "FC5", "FC3", "FC1", "FCZ", "FC2", "FC4", "FC6", "FT8"], 0.42),
        (["T7", "C5", "C3", "C1", "CZ", "C2", "C4", "C6", "T8"], 0.20),
        (["TP7", "CP5", "CP3", "CP1", "CPZ", "CP2", "CP4", "CP6", "TP8"], -0.02),
        (["P7", "P5", "P3", "P1", "PZ", "P2", "P4", "P6", "P8"], -0.25),
        (["PO7", "PO5", "PO3", "POZ", "PO4", "PO6", "PO8"], -0.50),
        (["CB1", "O1", "OZ", "O2", "CB2"], -0.72),
    ]
    coords = {}
    for names, y in rows:
        n = len(names)
        xs = np.linspace(-1.0, 1.0, n) if n > 1 else np.array([0.0])
        for name, x in zip(names, xs):
            coords[name] = (float(x), float(y))
    return np.array([coords[c] for c in CHANNEL_NAMES], dtype=np.float32)

def build_regions():
    regions = defaultdict(list)
    coords = build_scalp_coords()
    for i, name in enumerate(CHANNEL_NAMES):
        x, y = coords[i]
        if abs(x) > 0.6 and -0.10 < y < 0.50:
            key = "temporal_left" if x < 0 else "temporal_right"
        elif y >= 0.55:
            key = "frontal"
        elif y >= 0.10:
            key = "central"
        elif y >= -0.35:
            key = "parietal"
        else:
            key = "occipital"
        regions[key].append(i)
    return {k: np.array(v, dtype=np.int64) for k, v in regions.items()}

SCALP_COORDS = build_scalp_coords()
REGIONS = build_regions()


# =====================================================================
# DATA LOADING + NORMALIZATION
# =====================================================================
def load_seed_iv():
    Xs, ys, subs, sess, tris = [], [], [], [], []
    for session in sorted(TRIAL_LABELS_BY_SESSION.keys()):
        if session in SESSIONS_TO_SKIP: continue
        sdir = os.path.join(DATA_DIR, str(session))
        if not os.path.isdir(sdir): continue
        labels = TRIAL_LABELS_BY_SESSION[session]
        for fname in sorted(os.listdir(sdir)):
            if not fname.endswith(".mat"): continue
            subject = int(fname.split("_")[0])
            mat = sio.loadmat(os.path.join(sdir, fname))
            for t in range(1, TRIALS_PER_SESSION + 1):
                dk, pk = f"{DE_KEY_PREFIX}{t}", f"{PSD_KEY_PREFIX}{t}"
                if dk not in mat or pk not in mat: continue
                de = np.transpose(np.asarray(mat[dk], dtype=np.float32), (1, 0, 2))
                psd = np.transpose(np.asarray(mat[pk], dtype=np.float32), (1, 0, 2))
                psd = np.log(np.maximum(psd, 1e-10))
                feat = np.concatenate([psd, de], axis=2)
                n = feat.shape[0]
                Xs.append(feat)
                ys.append(np.full(n, labels[t - 1], dtype=np.int64))
                subs.append(np.full(n, subject, dtype=np.int64))
                sess.append(np.full(n, session, dtype=np.int64))
                tris.append(np.full(n, t, dtype=np.int64))
    X = np.concatenate(Xs, axis=0)
    return (X, np.concatenate(ys), np.concatenate(subs),
            np.concatenate(sess), np.concatenate(tris))

def select_features(X):
    if FEATURE_TYPE == "de": return X[:, :, 5:]
    if FEATURE_TYPE == "psd": return X[:, :, :5]
    return X

def normalize(X, subs, sess):
    Xn = X.copy()
    keys = subs * 100 + sess
    for k in np.unique(keys):
        m = keys == k
        blk = Xn[m]
        mu = blk.mean(axis=0, keepdims=True)
        sd = blk.std(axis=0, keepdims=True) + 1e-6
        Xn[m] = (blk - mu) / sd
    return Xn


# =====================================================================
# STAGE A : SINGLE STMAE
# =====================================================================
class ScalpPositionalEncoding(nn.Module):
    def __init__(self, coords, d_model):
        super().__init__()
        self.register_buffer("coords", torch.tensor(coords, dtype=torch.float32))
        self.mlp = nn.Sequential(nn.Linear(2, d_model), nn.GELU(), nn.Linear(d_model, d_model))

    def forward(self, x):
        return x + self.mlp(self.coords).unsqueeze(0)

class STMAE(nn.Module):
    def __init__(self, in_feat, d_model=AUTOENCODER_HIDDEN_SIZE, latent_dim=EMBEDDING_SIZE, 
                 heads=AUTOENCODER_HEADS, layers=AUTOENCODER_LAYERS):
        super().__init__()
        self.proj = nn.Linear(in_feat, d_model)
        self.pos = ScalpPositionalEncoding(SCALP_COORDS, d_model)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.mask_token, std=0.02)
        
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=heads, dim_feedforward=d_model * 4,
            dropout=0.1, batch_first=True, norm_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.to_latent = nn.Linear(d_model, latent_dim)
        self.latent_norm = nn.LayerNorm(latent_dim)
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, d_model), nn.GELU(), nn.Linear(d_model, in_feat)
        )

    def encode(self, x, mask=None):
        h = self.proj(x)
        if mask is not None:
            h = torch.where(mask.unsqueeze(-1), self.mask_token.expand_as(h), h)
        h = self.pos(h)
        h = self.encoder(h)
        return self.latent_norm(self.to_latent(h))

    def forward(self, x, mask):
        z = self.encode(x, mask)
        return self.decoder(z), z


def sample_mask(batch_size, device):
    mask = torch.zeros(batch_size, NUM_CHANNELS, dtype=torch.bool, device=device)
    region_keys = list(REGIONS.keys())
    for b in range(batch_size):
        if random.random() < REGION_MASK_CHANCE:
            k = random.choice([1, 2])
            for key in random.sample(region_keys, k):
                mask[b, torch.tensor(REGIONS[key], device=device)] = True
        else:
            n = max(1, int(RANDOM_MASK_FRACTION * NUM_CHANNELS))
            idx = torch.randperm(NUM_CHANNELS, device=device)[:n]
            mask[b, idx] = True
    return mask


def pretrain_stmae(X_pt, epochs=AUTOENCODER_EPOCHS, lr=AUTOENCODER_LR, batch=AUTOENCODER_BATCH_SIZE):
    in_feat = X_pt.shape[2]
    model = STMAE(in_feat).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    
    n = X_pt.shape[0]
    model.train()
    for ep in range(1, epochs + 1):
        perm = torch.randperm(n)
        tot, nb = 0.0, 0
        for i in range(0, n, batch):
            xb = X_pt[perm[i:i + batch]] # VRAM SLICING
            mask = sample_mask(xb.shape[0], DEVICE)
            recon, _ = model(xb, mask)
            m = mask.unsqueeze(-1).expand_as(xb)
            loss = F.mse_loss(recon[m], xb[m])
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            opt.step()
            tot += loss.item()
            nb += 1
        sched.step()
        if ep == 1 or ep % 5 == 0:
            print("  AE ep%03d masked_recon_mse=%.5f" % (ep, tot / max(nb, 1)))
    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "stmae_global.pt"))
    return model


# =====================================================================
# STAGE B : SEQUENCE BUILDER
# =====================================================================
def build_sequence_index(y, subs, sess, tri, seq_len=WINDOW_LENGTH, stride=1):
    keys = subs * 1000000 + sess * 10000 + tri
    seqs, labs, s_sub, s_ses, s_tri = [], [], [], [], []
    order = np.argsort(keys, kind="stable")
    for k in np.unique(keys):
        rows = order[keys[order] == k]
        if rows.shape[0] < seq_len: continue
        for start in range(0, rows.shape[0] - seq_len + 1, stride):
            win = rows[start:start + seq_len]
            seqs.append(win)
            labs.append(y[win[0]])
            s_sub.append(subs[win[0]])
            s_ses.append(sess[win[0]])
            s_tri.append(tri[win[0]])
    return (np.asarray(seqs, dtype=np.int64), np.asarray(labs, dtype=np.int64),
            np.asarray(s_sub, dtype=np.int64), np.asarray(s_ses, dtype=np.int64),
            np.asarray(s_tri, dtype=np.int64))


# =====================================================================
# STAGE C : MSC-TimesNet
# =====================================================================
def fft_topk_periods(x, k=TOP_FREQUENCIES):
    B, T, d = x.shape
    xf = torch.fft.rfft(x, dim=1)
    amp = xf.abs().mean(dim=2)
    amp[:, 0] = 0.0
    k = min(k, max(amp.shape[1] - 1, 1))
    _, idx = torch.topk(amp, k, dim=1)
    freqs = idx.float().mean(dim=0).round().long().clamp(min=1)
    periods = [max(int(T // f.item()), 1) for f in freqs]
    weights = torch.stack([amp[:, i] for i in freqs], dim=1)
    return periods, F.softmax(weights, dim=1)


class MultiScaleConvBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        h = max(d_model // 4, 8)
        self.b1 = nn.Sequential(nn.Conv2d(d_model, h, 1), nn.BatchNorm2d(h), nn.GELU())
        self.b3 = nn.Sequential(nn.Conv2d(d_model, h, 3, padding=1), nn.BatchNorm2d(h), nn.GELU())
        self.b5 = nn.Sequential(nn.Conv2d(d_model, h, 5, padding=2), nn.BatchNorm2d(h), nn.GELU())
        self.bp = nn.Sequential(
            nn.AvgPool2d(3, stride=1, padding=1), nn.Conv2d(d_model, h, 1),
            nn.BatchNorm2d(h), nn.GELU(),
        )
        self.fuse = nn.Sequential(nn.Conv2d(4 * h, d_model, 1), nn.BatchNorm2d(d_model))

    def forward(self, x):
        return self.fuse(torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1))


class TimesBlock(nn.Module):
    def __init__(self, d_model, topk=TOP_FREQUENCIES):
        super().__init__()
        self.topk = topk
        self.conv = MultiScaleConvBlock(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        B, T, d = x.shape
        periods, weights = fft_topk_periods(x, self.topk)
        outs = []
        for p in periods:
            pad = (math.ceil(T / p) * p) - T
            xp = F.pad(x, (0, 0, 0, pad)) if pad > 0 else x
            Tp = xp.shape[1]
            num_p = Tp // p
            z = xp.permute(0, 2, 1).reshape(B, d, num_p, p)
            z = self.conv(z)
            z = z.reshape(B, d, Tp).permute(0, 2, 1)[:, :T, :]
            outs.append(z)
        stacked = torch.stack(outs, dim=-1)
        w = weights.unsqueeze(1).unsqueeze(1)
        agg = (stacked * w).sum(dim=-1)
        return self.norm(agg + x)


class MSCTimesNet(nn.Module):
    def __init__(self, in_dim, d_model=CLASSIFIER_HIDDEN_SIZE, blocks=CLASSIFIER_BLOCKS, num_classes=NUM_EMOTIONS, dropout=CLASSIFIER_DROPOUT):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(in_dim, d_model), nn.LayerNorm(d_model))
        self.blocks = nn.ModuleList([TimesBlock(d_model) for _ in range(blocks)])
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=CLASSIFIER_HEADS, dim_feedforward=CLASSIFIER_FEEDFORWARD_SIZE,
            dropout=dropout, batch_first=True, norm_first=True, activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=1)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Dropout(dropout), nn.Linear(d_model, num_classes)
        )

    def forward(self, x):
        h = self.inp(x)
        for blk in self.blocks: h = blk(h)
        h = self.transformer(h)
        return self.head(h.mean(dim=1))


class EndToEndModel(nn.Module):
    def __init__(self, stmae, in_feat, finetune_encoder=False):
        super().__init__()
        self.stmae = stmae
        self.finetune_encoder = finetune_encoder
        self.net = MSCTimesNet(NUM_CHANNELS * EMBEDDING_SIZE)
        self.set_encoder_trainable(finetune_encoder)

    def set_encoder_trainable(self, flag):
        for p in self.stmae.parameters():
            p.requires_grad = bool(flag)

    def encode_seq(self, x):
        B, T, C, Fq = x.shape
        flat = x.reshape(B * T, C, Fq)
        if self.finetune_encoder and self.training:
            z = self.stmae.encode(flat)
        else:
            with torch.no_grad():
                z = self.stmae.encode(flat)
        return z.reshape(B, T, C * EMBEDDING_SIZE)

    def forward(self, x):
        return self.net(self.encode_seq(x))


# =====================================================================
# FAST GPU TRAINING UTILITIES
# =====================================================================
def balanced_weights(y):
    cnt = np.bincount(y, minlength=NUM_EMOTIONS).astype(np.float32)
    cnt[cnt == 0] = 1.0
    w = cnt.sum() / (NUM_EMOTIONS * cnt)
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)

def lr_at(ep, base_lr, warmup_epochs=None):
    if warmup_epochs is None: warmup_epochs = globals().get("WARMUP_EPOCHS", 3)
    warmup_epochs = max(1, int(warmup_epochs))
    if ep <= warmup_epochs: return base_lr * ep / warmup_epochs
    prog = (ep - warmup_epochs) / max(1, MAX_EPOCHS - warmup_epochs)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * min(prog, 1.0)))

def grouped_split(labels, groups, frac=VALIDATION_FRACTION, seed=RANDOM_SEED):
    gss = GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
    tr, va = next(gss.split(np.zeros(len(labels)), labels, groups))
    return tr, va

def make_batches(n, batch, shuffle=True):
    idx = np.random.permutation(n) if shuffle else np.arange(n)
    for i in range(0, n, batch): yield idx[i:i + batch]

def apply_channel_dropout(xb, p=CHANNEL_DROPOUT_RATE):
    if p <= 0: return xb
    B = xb.shape[0]
    keep = (torch.rand(B, 1, NUM_CHANNELS, 1, device=xb.device) > p).float()
    return xb * keep

def mixup(xb, yb, alpha=MIXUP_STRENGTH):
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(xb.shape[0], device=xb.device)
    return lam * xb + (1 - lam) * xb[perm], yb, yb[perm], lam

@torch.no_grad()
def adabn_recalibrate(model, X_pt, seq_idx_pt, rows_pt, batch=BATCH_SIZE):
    had_bn = any(isinstance(m, nn.BatchNorm2d) for m in model.modules())
    if not had_bn: return model
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.reset_running_stats()
            m.momentum = None
            m.train()
    for b_rows in make_batches(len(rows_pt), batch, shuffle=False):
        xb = X_pt[seq_idx_pt[rows_pt[b_rows]]]
        model.net(model.encode_seq(xb))
    model.eval()
    return model

@torch.no_grad()
def predict_probs(model, X_pt, seq_idx_pt, rows_pt, batch=BATCH_SIZE):
    model.eval()
    out = []
    for b_rows in make_batches(len(rows_pt), batch, shuffle=False):
        xb = X_pt[seq_idx_pt[rows_pt[b_rows]]]
        out.append(F.softmax(model(xb), dim=1).cpu().numpy())
    return np.concatenate(out, axis=0)

def fit(stmae, X_pt, seq_idx_pt, labels, groups, train_rows, seed):
    seed_everything(seed)
    in_feat = X_pt.shape[2]
    model = EndToEndModel(stmae, in_feat, finetune_encoder=True).to(DEVICE)
    tr_loc, va_loc = grouped_split(labels[train_rows], groups[train_rows], seed=seed)
    
    tr_rows_pt = torch.tensor(train_rows[tr_loc], dtype=torch.long, device=DEVICE)
    va_rows_pt = torch.tensor(train_rows[va_loc], dtype=torch.long, device=DEVICE)
    
    w = balanced_weights(labels[train_rows[tr_loc]])
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=LABEL_SMOOTHING)
    opt = torch.optim.AdamW([
        {"params": model.net.parameters(), "lr": LEARNING_RATE},
        {"params": model.stmae.parameters(), "lr": LEARNING_RATE * ENCODER_LR_SCALE},
    ], weight_decay=WEIGHT_DECAY)

    best_f1, best_state, bad = -1.0, None, 0
    for ep in range(1, MAX_EPOCHS + 1):
        cur = lr_at(ep, LEARNING_RATE)
        for i, g in enumerate(opt.param_groups):
            g["lr"] = cur * (ENCODER_LR_SCALE if i == 1 else 1.0)

        model.train()
        for b_rows in make_batches(len(tr_rows_pt), BATCH_SIZE):
            b_idx = tr_rows_pt[b_rows]
            xb = X_pt[seq_idx_pt[b_idx]]
            yb = torch.tensor(labels[tr_rows_pt[b_rows].cpu().numpy()], dtype=torch.long, device=DEVICE)
            
            xb = apply_channel_dropout(xb)
            xb, ya, ybb, lam = mixup(xb, yb)
            logits = model(xb)
            loss = lam * crit(logits, ya) + (1 - lam) * crit(logits, ybb)
                
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRADIENT_CLIP)
            opt.step()

        vp = predict_probs(model, X_pt, seq_idx_pt, va_rows_pt)
        vf1 = f1_score(labels[va_rows_pt.cpu().numpy()], vp.argmax(1), average="macro")
        if vf1 > best_f1:
            best_f1, bad = vf1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if ep >= MIN_EPOCHS and bad >= PATIENCE_EPOCHS:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_f1

def trial_level_scores(probs, y, sub, ses, tri):
    keys = sub * 1000000 + ses * 10000 + tri
    yt, yp = [], []
    for k in np.unique(keys):
        m = keys == k
        yt.append(y[m][0])
        yp.append(probs[m].mean(axis=0).argmax())
    yt, yp = np.array(yt), np.array(yp)
    return accuracy_score(yt, yp), f1_score(yt, yp, average="macro"), len(yt)


# =====================================================================
# EVALUATION OF ONE FOLD
# =====================================================================
def run_fold(protocol, fold_name, stmae, X_pt, seq_idx_pt, packs, train_rows, test_rows):
    labels, s_sub, s_ses, s_tri = packs[1:]
    groups = s_sub * 1000000 + s_ses * 10000 + s_tri
    seeds = SEEDS_PER_PROTOCOL[protocol]
    te_rows_pt = torch.tensor(test_rows, dtype=torch.long, device=DEVICE)

    prob_sum = None
    for sd in seeds:
        stmae_fold = copy.deepcopy(stmae) 
        model, _ = fit(stmae_fold, X_pt, seq_idx_pt, labels, groups, train_rows, sd)
        if RECALIBRATE_BATCHNORM:
            model = adabn_recalibrate(model, X_pt, seq_idx_pt, te_rows_pt)
        p = predict_probs(model, X_pt, seq_idx_pt, te_rows_pt)
        prob_sum = p if prob_sum is None else prob_sum + p
        
    probs = prob_sum / len(seeds)
    yt = labels[test_rows]
    window_acc = accuracy_score(yt, probs.argmax(1))
    window_f1 = f1_score(yt, probs.argmax(1), average="macro")
    trial_acc, trial_f1, num_trials = trial_level_scores(probs, yt, s_sub[test_rows], s_ses[test_rows], s_tri[test_rows])

    print("  -> %-18s %-22s windows: %.4f acc / %.4f f1  |  trials: %.4f acc / %.4f f1"
          "  (%d windows, %d trials)"
          % (protocol, fold_name, window_acc, window_f1, trial_acc, trial_f1, len(test_rows), num_trials))
          
    REPORT.append(dict(protocol=protocol, fold=fold_name, window_acc=window_acc, window_f1=window_f1, 
                       trial_acc=trial_acc, trial_f1=trial_f1, num_windows=len(test_rows), num_trials=num_trials))


def folds_for(protocol, s_sub, s_ses):
    out = []
    if protocol == "loso_subject":
        for sb in np.unique(s_sub):
            te = np.where(s_sub == sb)[0]
            tr = np.where(s_sub != sb)[0]
            out.append(("subject_%d" % sb, tr, te))
    return out


# =====================================================================
# MAIN -- LOSO EVALUATION
# =====================================================================
def run_evaluation_block(block_label, protocol, stmae, X_pt, seq_idx_pt, packs):
    seq_idx, labels, s_sub, s_ses, s_tri = packs

    print("\n" + "-" * 84)
    print("- %s -- protocol=%s" % (block_label, protocol))
    print("-" * 84)

    print("\nfold plan -- check the fit count before letting this run")
    fold_list = folds_for(protocol, s_sub, s_ses)
    num_seeds = len(SEEDS_PER_PROTOCOL[protocol])
    total_fits = len(fold_list) * num_seeds
    print("  %-18s %2d folds x %d seeds = %3d fits" % (protocol, len(fold_list), num_seeds, total_fits))
    
    for fold_name, tr_, te_ in fold_list:
        overlap = set(s_sub[tr_].tolist()) & set(s_sub[te_].tolist())
        assert not overlap, ("SUBJECT LEAK in %s/%s: %s" % (protocol, fold_name, overlap))
        
    print("  subject-independence checked OK")
    print("  TOTAL MODEL FITS = %d" % total_fits)

    print("\nrunning %s" % block_label)
    report_start = len(REPORT)
    for fold_name, tr_rows, te_rows in fold_list:
        run_fold(protocol, fold_name, stmae, X_pt, seq_idx_pt, packs, tr_rows, te_rows)
    block_rows = REPORT[report_start:]

    df = pd.DataFrame(block_rows)
    results_path = os.path.join(OUTPUT_DIR, "results_%s.csv" % protocol)
    df.to_csv(results_path, index=False)
    print("\nsaved: %s" % results_path)

    print("\n" + "=" * 84)
    print("SUMMARY -- %s (protocol=%s)" % (block_label, protocol))
    print("=" * 84)
    if not df.empty:
        summary = (df.groupby("protocol")
                     .agg(window_acc=("window_acc", "mean"), window_sd=("window_acc", "std"),
                          window_f1=("window_f1", "mean"), trial_acc=("trial_acc", "mean"),
                          trial_f1=("trial_f1", "mean"), folds=("window_acc", "size"))
                     .reset_index())
        print(summary.to_string(index=False, float_format=lambda v: "%.4f" % v))
    return df


def main():
    t0 = time.time()
    seed_everything(RANDOM_SEED)
    print("device:", DEVICE)
    print("Scalp regions:", {k: len(v) for k, v in REGIONS.items()})

    print("\n[1] loading data")
    X_raw, y, subs, sess, tri = load_seed_iv()
    X = select_features(X_raw)
    print("shape=%s  features_per_channel=%d  labels=%s" % (X.shape, X.shape[2], np.bincount(y)))
    print("subjects:", sorted(np.unique(subs).tolist()))
    print("sessions:", sorted(np.unique(sess).tolist()))

    print("\n[2] normalizing (%s)" % NORMALIZATION_METHOD)
    X = normalize(X, subs, sess)
    
    print("\n[2.5] pushing dataset to VRAM for fast execution...")
    X_pt = torch.tensor(X, dtype=torch.float32, device=DEVICE)

    print("\n[3] pretraining the spatial masked autoencoder (STMAE) globally")
    stmae = pretrain_stmae(X_pt)
    for p in stmae.parameters():
        p.requires_grad = False

    print("\n[4] building sliding windows")
    packs = build_sequence_index(y, subs, sess, tri, WINDOW_LENGTH, stride=1)
    seq_idx = packs[0]
    seq_idx_pt = torch.tensor(seq_idx, dtype=torch.long, device=DEVICE)
    print("  %d windows  labels=%s" % (seq_idx.shape[0], np.bincount(packs[1])))

    print("\nfeatures=%s  normalization=%s  sessions_skipped=%s  batchnorm_recalibration=%s"
          % (FEATURE_TYPE, NORMALIZATION_METHOD, SESSIONS_TO_SKIP, RECALIBRATE_BATCHNORM))

    # -----------------------------------------------------------------
    # EVALUATION -- leave-one-subject-out
    # -----------------------------------------------------------------
    if RUN_LOSO:
        run_evaluation_block("EVALUATION: LOSO", "loso_subject", stmae, X_pt, seq_idx_pt, packs)
    else:
        print("\n[EVALUATION: LOSO] skipped (RUN_LOSO = False)")

    print("\nWall time: %.1f min" % ((time.time() - t0) / 60.0))


if __name__ == "__main__":
    main()

device: cuda
Scalp regions: {'frontal': 14, 'temporal_left': 6, 'central': 10, 'temporal_right': 6, 'parietal': 14, 'occipital': 12}

[1] loading data
shape=(37575, 62, 10)  features_per_channel=10  labels=[10170 10245  9225  7935]
subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
sessions: [1, 2, 3]

[2] normalizing (subject_session)

[2.5] pushing dataset to VRAM for fast execution...

[3] pretraining the spatial masked autoencoder (STMAE) globally
  AE ep001 masked_recon_mse=0.44195
  AE ep005 masked_recon_mse=0.22090
  AE ep010 masked_recon_mse=0.19562
  AE ep015 masked_recon_mse=0.18535
  AE ep020 masked_recon_mse=0.17877
  AE ep025 masked_recon_mse=0.17608
  AE ep030 masked_recon_mse=0.17405

[4] building sliding windows
  27855 windows  labels=[7740 7815 6795 5505]

features=both  normalization=subject_session  sessions_skipped=[]  batchnorm_recalibration=True

------------------------------------------------------------------------------------
- EVALUATION: LOSO -- 